# 08 — Final Elo Model

Clean Elo system built on `02_elo_ratings.ipynb` with two additions:
- **Time decay** — recent matches weighted more heavily (8 year half-life)
- **Pre-WC elite boosts** — top teams in weaker confederations get +30 Elo before each WC they qualify for

No confederation multipliers, no field strength, no competition ratings — just clean Elo.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 1. Load data

In [ ]:
df = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\results_clean.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Total matches: {len(df)}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

## 2. Pre-WC elite boosts

Top 1-2 teams in weaker confederations get +30 Elo before each WC they qualify for.
Permanent and carried forward into all future matches.

In [ ]:
ELITE_BOOST = 30

elite_teams = {
    # CAF
    'Morocco':       [2002, 2006, 2010, 2014, 2018, 2022, 2026],
    'Senegal':       [2002, 2006, 2010, 2014, 2018, 2022, 2026],
    # AFC
    'Japan':         [2002, 2006, 2010, 2014, 2018, 2022, 2026],
    'South Korea':   [2002, 2006, 2010, 2014, 2018, 2022, 2026],
    # CONCACAF
    'Mexico':        [2002, 2006, 2010, 2014, 2018, 2022, 2026],
    'United States': [2002, 2006, 2010, 2014, 2018, 2022, 2026],
    # OFC
    'New Zealand':   [2002, 2006, 2010],
}

# First match date per WC year
wc_matches = df[df['tournament'] == 'FIFA World Cup'].copy()
wc_matches['year'] = wc_matches['date'].dt.year
wc_starts = wc_matches.groupby('year')['date'].min().to_dict()

print("WC start dates & boosts:")
for y, d in sorted(wc_starts.items()):
    teams = [t for t, yrs in elite_teams.items() if y in yrs]
    print(f"  {y}: {d.date()} → {teams}")

## 3. Elo system

In [ ]:
STARTING_ELO   = 1500
HOME_ADVANTAGE = 100
max_date       = df['date'].max()

elo_ratings    = {}
boosts_applied = set()

def get_elo(team):
    if team not in elo_ratings:
        elo_ratings[team] = STARTING_ELO
    return elo_ratings[team]

def expected_score(ra, rb):
    return 1 / (1 + 10 ** ((rb - ra) / 400))

def apply_pre_wc_boosts(current_date):
    for wc_year, start_date in wc_starts.items():
        if current_date == start_date:
            for team, years in elite_teams.items():
                if wc_year in years and (team, wc_year) not in boosts_applied:
                    if team in elo_ratings:
                        elo_ratings[team] += ELITE_BOOST
                        boosts_applied.add((team, wc_year))
                        print(f"  +{ELITE_BOOST} → {team} (WC {wc_year})")

def update_elo(ht, at, hs, as_, tw, neutral, date):
    he, ae = get_elo(ht), get_elo(at)
    ha     = he if neutral else he + HOME_ADVANTAGE
    hexp   = expected_score(ha, ae)
    aexp   = expected_score(ae, ha)

    if hs > as_:   hact, aact = 1, 0
    elif hs < as_: hact, aact = 0, 1
    else:          hact, aact = 0.5, 0.5

    # Time decay — 8 year half life
    days_ago = (max_date - pd.to_datetime(date)).days
    decay    = np.exp(-days_ago / (8 * 365))

    # Era-based K
    year = pd.to_datetime(date).year
    bk   = 40 if year >= 2022 else 35 if year >= 2019 else 25 if year >= 2014 else 20

    k = bk * tw * (0.3 + 0.7 * decay)

    elo_ratings[ht] = he + k * (hact - hexp)
    elo_ratings[at] = ae + k * (aact - aexp)

print("Processing matches...")
print()

for _, row in df.iterrows():
    apply_pre_wc_boosts(row['date'])
    update_elo(
        row['home_team'], row['away_team'],
        row['home_score'], row['away_score'],
        row['tournament_weight'], row['neutral'], row['date']
    )

print(f"Done. {len(elo_ratings)} teams rated.")
print(f"Elite boosts applied: {len(boosts_applied)}")

## 4. Normalize & view rankings

In [ ]:
# Normalize: mean=1500, std=150
all_elos     = pd.Series(list(elo_ratings.values()))
current_mean = all_elos.mean()
current_std  = all_elos.std()
TARGET_MEAN, TARGET_STD = 1500, 150

for team in elo_ratings:
    elo_ratings[team] = TARGET_MEAN + (elo_ratings[team] - current_mean) * (TARGET_STD / current_std)

elo_df = pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo'])
elo_df = elo_df.sort_values('elo', ascending=False).reset_index(drop=True)
elo_df.index += 1

print("Global top 20:")
print(elo_df.head(20).to_string())

## 5. Save

In [ ]:
gs = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\group_stages_clean.csv")
wc_teams = gs['nation'].tolist()
wc_elo = elo_df[elo_df['team'].isin(wc_teams)].copy()
wc_elo.index = range(1, len(wc_elo) + 1)

print(f"All {len(wc_elo)} WC 2026 teams ranked:")
print(wc_elo.to_string())

# Save WC teams
wc_elo.to_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\elo_ratings.csv", index=False)

# Save all teams (needed for feature engineering)
elo_df.to_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\elo_all_teams.csv", index=False)

print("\nSaved:")
print("  elo_ratings.csv      — WC 2026 teams")
print("  elo_all_teams.csv    — all teams (for feature engineering)")